<a href="https://colab.research.google.com/github/cyruslayo/nba_bot/blob/main/notebooks/nba_market_models_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NBA Market Models — Colab Training Notebook

Trains the `spread`, `total`, and `first_half` classification models used by `nba_bot`.

This notebook mirrors the repository training pipeline in `nba_bot/train.py` and uses the same feature builders from `nba_bot/features.py`.

Artifacts are saved to Google Drive for later use by `nba-bot-scan`.

In [ ]:
# Cell 1 — Install dependencies and force restart
import os

if os.path.exists('/content/.nba_bot_installed'):
    print('Dependencies already installed. Skipping install.')
else:
    print('Installing dependencies...')
    !pip install "numpy==1.26.4" "pandas==2.2.2" "xgboost==2.0.3" scikit-learn joblib tqdm nba_api "nba-on-court>=0.2.0,<1.0" -q

    if not os.path.exists('/content/nba_bot'):
        !git clone https://github.com/cyruslayo/nba_bot.git /content/nba_bot
    else:
        !git -C /content/nba_bot pull

    !pip install -e /content/nba_bot --no-deps -q
    !touch /content/.nba_bot_installed

    print('Installed pinned CPU training stack. Restarting kernel...')
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_OUTPUT = '/content/drive/MyDrive/nba_bot/'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print('Output dir:', DRIVE_OUTPUT)

In [ ]:
# Cell 2 — Verify imports and runtime capabilities
import numpy as np
import pandas as pd
import xgboost as xgb

print('numpy:', np.__version__)
print('pandas:', pd.__version__)
print('XGBoost:', xgb.__version__)
print()
print('Note: Colab xgboost wheels are typically CPU-only in this setup.')
print('GPU will not be used for xgboost training here.')
print('The fix is to reduce RAM pressure, not force CUDA.')

!nvidia-smi --query-gpu=name,memory.total,utilization.gpu --format=csv,noheader 2>/dev/null || echo 'No GPU detected'

print('✅ Imports successful')

In [ ]:
# Cell 3 — Training config
SEASONS = [2021, 2022, 2023, 2024]
MARKET_TYPES = ['spread', 'total', 'first_half']
USE_ADVANCED_FEATURES = True
SAMPLE_GAMES_PER_SEASON = None  # kept for compatibility with the data-load cell
TEST_SIZE = 0.2
RANDOM_STATE = 42
USE_GPU = False  # Colab xgboost wheel is CPU-only in this setup

print('Seasons:', SEASONS)
print('Market types:', MARKET_TYPES)
print('Advanced features:', USE_ADVANCED_FEATURES)
print('Sample games/season:', SAMPLE_GAMES_PER_SEASON)
print('Use GPU:', USE_GPU)
print('Training mode: disk-backed CPU hist')

In [ ]:
# Cell 4 — Load play-by-play data
import os
import pandas as pd
import nba_on_court as noc

def fix_columns(df):
    if df is None:
        return None
    df.columns = [c.upper() for c in df.columns]
    if 'GAME_ID' not in df.columns:
        possible_id_cols = [c for c in df.columns if 'NBASTATS' in c or 'GAMEID' in c or 'GAME_ID' in c]
        if possible_id_cols:
            df = df.rename(columns={possible_id_cols[0]: 'GAME_ID'})
    return df

def load_data_safe(seasons, data_type):
    try:
        df = noc.load_nba_data(seasons=seasons, data=data_type)
        if df is not None and len(df) > 0:
            return fix_columns(df)
    except Exception as e:
        print(f'Library load failed for {data_type}: {e}')

    dfs = []
    for season in seasons:
        fpath = f'/content/{data_type}_{season}.tar.xz'
        if os.path.exists(fpath):
            raw = pd.read_csv(fpath, compression='xz')
            dfs.append(fix_columns(raw))
    return pd.concat(dfs, ignore_index=True) if dfs else None

df_nbastats = load_data_safe(SEASONS, 'nbastats')
if df_nbastats is None or df_nbastats.empty:
    raise ValueError('Could not load nbastats data')

df_pbpstats = None
if USE_ADVANCED_FEATURES:
    df_pbpstats = load_data_safe(SEASONS, 'pbpstats')

print(f'nbastats rows: {len(df_nbastats):,}')
print(f'nbastats games: {df_nbastats["GAME_ID"].nunique():,}')
if df_pbpstats is not None:
    print(f'pbpstats rows: {len(df_pbpstats):,}')
    print(f'pbpstats games: {df_pbpstats["GAME_ID"].nunique():,}')

if SAMPLE_GAMES_PER_SEASON:
    game_ids = df_nbastats['GAME_ID'].dropna().astype(str)
    season_prefix = game_ids.str[:4]
    sampled_ids = []
    for prefix in sorted(season_prefix.unique()):
        ids = sorted(df_nbastats.loc[season_prefix == prefix, 'GAME_ID'].dropna().unique().tolist())
        sampled_ids.extend(ids[:SAMPLE_GAMES_PER_SEASON])
    df_nbastats = df_nbastats[df_nbastats['GAME_ID'].isin(sampled_ids)].copy()
    if df_pbpstats is not None:
        df_pbpstats = df_pbpstats[df_pbpstats['GAME_ID'].isin(sampled_ids)].copy()
    print(f'Sampled nbastats rows: {len(df_nbastats):,}')
    print(f'Sampled games: {df_nbastats["GAME_ID"].nunique():,}')

print('✅ Data load complete')

In [ ]:
# Cell 5 — Load team ratings for T2 features
import importlib.util
import sys

player_ratings = {}
if USE_ADVANCED_FEATURES:
    !sed -i "s/per_mode_simple/per_mode_detailed/g" /content/nba_bot/nba_bot/team_stats_cache.py

    config_path = '/content/nba_bot/nba_bot/config.py'
    config_spec = importlib.util.spec_from_file_location('nba_bot.config', config_path)
    config_mod = importlib.util.module_from_spec(config_spec)
    sys.modules['nba_bot.config'] = config_mod
    config_spec.loader.exec_module(config_mod)

    module_file = '/content/nba_bot/nba_bot/team_stats_cache.py'
    spec = importlib.util.spec_from_file_location('nba_bot.team_stats_cache', module_file)
    tsc_module = importlib.util.module_from_spec(spec)
    sys.modules['nba_bot.team_stats_cache'] = tsc_module
    spec.loader.exec_module(tsc_module)

    refresh_team_stats = tsc_module.refresh_team_stats
    player_ratings = refresh_team_stats(output_path='/content/team_stats.json')
    print(f'Loaded ratings for {len(player_ratings)} teams')
else:
    print('Skipping team ratings')

In [ ]:
# Cell 6 — Disk-backed training helpers (avoids RAM blow-ups)
import gc
import os
import json
import joblib
import numpy as np
import xgboost as xgb
import importlib.util
import sys
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

features_path = '/content/nba_bot/nba_bot/features.py'
spec = importlib.util.spec_from_file_location('nba_bot.features', features_path)
features_module = importlib.util.module_from_spec(spec)
sys.modules['nba_bot.features'] = features_module
spec.loader.exec_module(features_module)

build_spread_rows = features_module.build_spread_rows
build_total_rows = features_module.build_total_rows
build_first_half_rows = features_module.build_first_half_rows

from nba_bot.config import FEATURES_SPREAD, FEATURES_TOTAL, FEATURES_FIRST_HALF

WORK_DIR = '/content/nba_market_training'
os.makedirs(WORK_DIR, exist_ok=True)
results = {}

def _market_builder(market_type):
    if market_type == 'spread':
        return build_spread_rows, FEATURES_SPREAD, 'covered_spread', 'xgb_spread_t2.pkl', 'feature_cols_spread.pkl'
    if market_type == 'total':
        return build_total_rows, FEATURES_TOTAL, 'went_over', 'xgb_total_t2.pkl', 'feature_cols_total.pkl'
    if market_type == 'first_half':
        return build_first_half_rows, FEATURES_FIRST_HALF, 'home_win', 'xgb_first_half_t2.pkl', 'feature_cols_first_half.pkl'
    raise ValueError(f'Unknown market_type: {market_type}')

def _iter_game_ids(df):
    ids = df['GAME_ID'].dropna().unique().tolist()
    ids.sort()
    return ids

def _game_split(game_id):
    text = str(game_id)
    total = sum(ord(ch) for ch in text)
    return 'valid' if total % 5 == 0 else 'train'

def build_market_memmaps(market_type):
    builder, feature_cols, target_col, _, _ = _market_builder(market_type)
    feature_count = len(feature_cols)
    game_ids = _iter_game_ids(df_nbastats)

    print('=' * 70)
    print(f'Counting rows for {market_type}')
    print('=' * 70)

    train_rows = 0
    valid_rows = 0

    for idx, game_id in enumerate(game_ids, start=1):
        game_nbastats = df_nbastats[df_nbastats['GAME_ID'] == game_id].copy()
        game_pbpstats = None
        if df_pbpstats is not None:
            game_pbpstats = df_pbpstats[df_pbpstats['GAME_ID'] == game_id].copy()

        game_df = builder(
            df_nbastats=game_nbastats,
            df_pbpstats=game_pbpstats,
            player_ratings=player_ratings,
        )
        if not game_df.empty:
            game_df = game_df[game_df['time_remaining'] < 2870].copy()
            game_df = game_df.dropna(subset=feature_cols + [target_col])
            rows = len(game_df)
            if _game_split(game_id) == 'train':
                train_rows += rows
            else:
                valid_rows += rows
        del game_nbastats, game_pbpstats, game_df
        if idx % 100 == 0:
            gc.collect()
            print(f'Counted {idx}/{len(game_ids)} games | train_rows={train_rows:,} | valid_rows={valid_rows:,}')

    print(f'Allocating memmaps for {market_type}: train={train_rows:,}, valid={valid_rows:,}')
    x_train_path = os.path.join(WORK_DIR, f'{market_type}_X_train.dat')
    y_train_path = os.path.join(WORK_DIR, f'{market_type}_y_train.dat')
    x_valid_path = os.path.join(WORK_DIR, f'{market_type}_X_valid.dat')
    y_valid_path = os.path.join(WORK_DIR, f'{market_type}_y_valid.dat')

    X_train = np.memmap(x_train_path, dtype=np.float32, mode='w+', shape=(train_rows, feature_count))
    y_train = np.memmap(y_train_path, dtype=np.int8, mode='w+', shape=(train_rows,))
    X_valid = np.memmap(x_valid_path, dtype=np.float32, mode='w+', shape=(valid_rows, feature_count))
    y_valid = np.memmap(y_valid_path, dtype=np.int8, mode='w+', shape=(valid_rows,))

    train_pos = 0
    valid_pos = 0

    print('=' * 70)
    print(f'Writing rows for {market_type}')
    print('=' * 70)

    for idx, game_id in enumerate(game_ids, start=1):
        game_nbastats = df_nbastats[df_nbastats['GAME_ID'] == game_id].copy()
        game_pbpstats = None
        if df_pbpstats is not None:
            game_pbpstats = df_pbpstats[df_pbpstats['GAME_ID'] == game_id].copy()

        game_df = builder(
            df_nbastats=game_nbastats,
            df_pbpstats=game_pbpstats,
            player_ratings=player_ratings,
        )
        if not game_df.empty:
            game_df = game_df[game_df['time_remaining'] < 2870].copy()
            game_df = game_df.dropna(subset=feature_cols + [target_col])
            if not game_df.empty:
                X_chunk = game_df[feature_cols].to_numpy(dtype=np.float32, copy=True)
                y_chunk = game_df[target_col].to_numpy(dtype=np.int8, copy=True)
                rows = len(game_df)
                if _game_split(game_id) == 'train':
                    X_train[train_pos:train_pos + rows] = X_chunk
                    y_train[train_pos:train_pos + rows] = y_chunk
                    train_pos += rows
                else:
                    X_valid[valid_pos:valid_pos + rows] = X_chunk
                    y_valid[valid_pos:valid_pos + rows] = y_chunk
                    valid_pos += rows
                del X_chunk, y_chunk
        del game_nbastats, game_pbpstats, game_df
        if idx % 100 == 0:
            X_train.flush(); y_train.flush(); X_valid.flush(); y_valid.flush()
            gc.collect()
            print(f'Wrote {idx}/{len(game_ids)} games | train_pos={train_pos:,} | valid_pos={valid_pos:,}')

    X_train.flush(); y_train.flush(); X_valid.flush(); y_valid.flush()
    del X_train, y_train, X_valid, y_valid
    gc.collect()

    return {
        'x_train_path': x_train_path,
        'y_train_path': y_train_path,
        'x_valid_path': x_valid_path,
        'y_valid_path': y_valid_path,
        'train_rows': train_rows,
        'valid_rows': valid_rows,
        'feature_cols': feature_cols,
        'target_col': target_col,
        'feature_count': feature_count,
    }

def train_market_from_memmaps(market_type, cache_info):
    _, feature_cols, _, model_name, feature_name = _market_builder(market_type)
    X_train = np.memmap(cache_info['x_train_path'], dtype=np.float32, mode='r', shape=(cache_info['train_rows'], cache_info['feature_count']))
    y_train = np.memmap(cache_info['y_train_path'], dtype=np.int8, mode='r', shape=(cache_info['train_rows'],))
    X_valid = np.memmap(cache_info['x_valid_path'], dtype=np.float32, mode='r', shape=(cache_info['valid_rows'], cache_info['feature_count']))
    y_valid = np.memmap(cache_info['y_valid_path'], dtype=np.int8, mode='r', shape=(cache_info['valid_rows'],))

    print('=' * 70)
    print(f'Training {market_type} on CPU-only XGBoost')
    print('=' * 70)
    print(f'Train rows: {cache_info["train_rows"]:,} | Valid rows: {cache_info["valid_rows"]:,}')

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method='hist',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        verbosity=1,
        n_jobs=2,
    )
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=True)

    probs = model.predict_proba(X_valid)[:, 1]
    metrics = {
        'log_loss': round(float(log_loss(y_valid, probs)), 4),
        'brier_score': round(float(brier_score_loss(y_valid, probs)), 4),
        'auc_roc': round(float(roc_auc_score(y_valid, probs)), 4),
    }

    model_path = os.path.join(DRIVE_OUTPUT, model_name)
    feature_path = os.path.join(DRIVE_OUTPUT, feature_name)
    joblib.dump(model, model_path)
    joblib.dump(feature_cols, feature_path)

    results[market_type] = {
        'metrics': metrics,
        'model_path': model_path,
        'feature_path': feature_path,
    }

    print('Metrics:', metrics)
    print('Saved model:', model_path)
    print('Saved features:', feature_path)

    del X_train, y_train, X_valid, y_valid, model, probs
    gc.collect()
    return results[market_type]

print('✅ Disk-backed helpers loaded')

In [ ]:
# Cell 7 — Build + train spread model (disk-backed)
if 'spread' in MARKET_TYPES:
    spread_cache = build_market_memmaps('spread')
    spread_result = train_market_from_memmaps('spread', spread_cache)
    print('Spread complete:', spread_result)
else:
    print('Skipping spread')

In [ ]:
# Cell 8 — Build + train total model (disk-backed)
if 'total' in MARKET_TYPES:
    total_cache = build_market_memmaps('total')
    total_result = train_market_from_memmaps('total', total_cache)
    print('Total complete:', total_result)
else:
    print('Skipping total')

In [ ]:
# Cell 9 — Build + train first-half model (disk-backed)
if 'first_half' in MARKET_TYPES:
    first_half_cache = build_market_memmaps('first_half')
    first_half_result = train_market_from_memmaps('first_half', first_half_cache)
    print('First-half complete:', first_half_result)
else:
    print('Skipping first_half')

In [ ]:
# Cell 10 — Export summary
print('Completed markets:', list(results.keys()))
print()
for market_type, info in results.items():
    print('=' * 70)
    print(market_type)
    print('metrics:', info['metrics'])
    print('model:', info['model_path'])
    print('features:', info['feature_path'])

print('')
print('Local env vars after downloading artifacts:')
print('  export NBA_BOT_SPREAD_MODEL_PATH=/path/to/xgb_spread_t2.pkl')
print('  export NBA_BOT_TOTAL_MODEL_PATH=/path/to/xgb_total_t2.pkl')
print('  export NBA_BOT_FIRST_HALF_MODEL_PATH=/path/to/xgb_first_half_t2.pkl')
print('  export NBA_BOT_ENABLE_SPREAD_TRADING=true')
print('  export NBA_BOT_ENABLE_TOTAL_TRADING=true')
print('  export NBA_BOT_ENABLE_FIRST_HALF_TRADING=true')
print('')
print('Notebook now uses disk-backed memmaps to avoid RAM crashes during spread/total expansion.')